In [424]:
# Import des utilitaires

## librairies de traitement de données
import pandas as pd
import numpy as np

## librairies de gestion de fichiers
import os

## librairies de visualisation des valeurs manquantes
import missingno as msno

## librairies de statistiques et de machine learning
import scipy.stats as st
from sklearn.linear_model import LinearRegression

## librairies de visualisation
import matplotlib.pyplot as plt
import seaborn as sns

In [425]:
# Fonctions pour importer tout type de fichier en determinant le type de fichier et son delimiter

##Fonction pour récupérer le séparateur
def separateurDetect(fichier):

    with open(fichier, mode= "r") as f: #encoding= encodage
        premieres_lignes = f.readline(15)
    for sep in [',', ';', "\t", ':', '|']:
        if sep in premieres_lignes:
            return sep
    return ',' # Par défaut

#Fonction pour importer un fichier
def importFichier(fichier):

    # Le fichier existe ?
    if not os.path.exists(fichier):
        raise FileExistsError(f"Le fichier {fichier} est introuvable.")
    
    # Détecton le type du fichier
    extension = os.path.splitext(fichier)[1].lower()

    if extension in ['.csv', '.tsv', '.txt']:
        # Détection du séparateur
        separateur = separateurDetect(fichier)
        print(f"Chargement d'un fichier avec pour séparateur '{separateur}'.")
        return pd.read_csv(fichier, sep= separateur)
    
    elif extension in ['.xls', '.xslx']:
        print(f"Chargement d'un fichier Excel .")#
        return pd.read_excel(fichier, engine= 'openpyxl')
    
    else:
        raise ValueError(f"Format de fichier non pris en compte : {extension}")


In [426]:
# Fonctions pour analyser les données

## Fonction pour catégoriser les variables
def categoriseVariables(df):
    
    info_type = df.dtypes
    variable_qualitative = []
    variable_quantitative = []

    for index in info_type.index:
        if (info_type[index] == 'float') | (info_type[index] == 'int'):
            variable_quantitative.append(index)
        else:
            variable_qualitative.append(index)
    return variable_qualitative, variable_quantitative

## Fonction pour décrire une dataframe 
def infoData(df):
    info = df.info().pd.DataFrame()
    return info

def descriptionData(df):
    nb_lignes, nb_col = df.shape
    print(f"La dataframe contient {nb_lignes} lignes et {nb_col} colonnes.")

## Fonction pour afficher les afficher les colonnes ayant un taux de remplissage inférieur à un seuil
def colManquantes(df, seuil):
    # Calculer le taux de remplissage 
    taux_remplissage = df.notnull().mean() * 100
    # Sélection des colonnes avec un taux de remplissage < seuil
    colonnes = taux_remplissage[taux_remplissage < seuil].index.tolist()
    print(f"nombre de colonnes avec un taux de remplissage inférieur à {seuil}% : {len(colonnes)}")
    return colonnes

## Fonction pour supprimer des colonnes dans une dataframe
def supprimeCol(df, colonnes):
    return df.drop(columns = colonnes)

## Fonction pour selectionner des colonnes dans une dataframe
def colSelection(df, colonnes):
    return df[colonnes]

In [427]:
# Fonctions pour remplacer les valeurs manquantes

## Remplacer les valeurs manquantes en utilisant le coeficient d'asymétrie pour choisir la méthode de remplacement
def remplacerValParStats(dataframe, colonnes):

    for colonne in colonnes:
        # Calculer le coefficient d'asymétrie
        skewness = st.skew(dataframe[colonne].dropna())
        #print(f"Coefficient d'asymétrie pour {colonne}: {skewness}")

        # Déterminer la méthode statistique appropriée
        if abs(skewness) < 0.5:
            # Distribution symétrique
            methode = 'moyenne'
            valeur_remplacement = dataframe[colonne].mean()
        else:
            # Asymétrie positive/négative
            methode = 'médiane'
            valeur_remplacement = dataframe[colonne].median()

        print(f"Coefficient d'asymétrie de skewness pour {colonne}: {skewness} , Méthode de remplacement utilisée : {methode}")

        # Remplacer les valeurs manquantes
        dataframe.loc[:, colonne] = dataframe[colonne].fillna(valeur_remplacement)

    return dataframe

# Fonction pour remplacer les NA d'une colonne par regréssion linéaire
def remplacerNaParRegression(data, colonne_a_predire, colonnes_predictives):

    # Sélection des données d'entrainement et de test
    data_train = data[data[colonne_a_predire].notnull()]
    data_test = data[data[colonne_a_predire].isnull()]

    # Création et entrainement du modèle de régression linéaire
    model = LinearRegression()
    model.fit(data_train[colonnes_predictives], data_train[colonne_a_predire])

    # Prédiction des valeurs manquantes
    valeurs_prédites = model.predict(data_test[colonnes_predictives])

    # Remplacement des valeurs manquantes par celles prédites
    data.loc[data[colonne_a_predire].isnull(), colonne_a_predire] = valeurs_prédites

    return data

In [428]:
# Fonctions pour gerer les valeurs aberrantes

## Remplacer les valeurs maximales supérieures à 3 fois l'écart-type par la médiane
def remplacerValeursMaxParMediane(dataframe, colonnes):
    for colonne in colonnes:
        # Calculer la moyenne et l'écart-type de la colonne
        moyenne = dataframe[colonne].mean()
        ecart_type = dataframe[colonne].std()
        
        # Définir une valeur seuil (Les valeurs qui se trouvent à plus de trois écarts-types de la moyenne sont considérées comme des valeurs aberrantes)
        seuil = moyenne + 3 * ecart_type
        
        # Calculer la médiane de la colonne
        mediane = dataframe[colonne].median()
        
        # Remplacer les valeurs supérieures au seuil par la médiane
        dataframe.loc[dataframe[colonne] > seuil, colonne] = mediane
    
    return dataframe

## Fonction pour remplacer les valeurs trop petites par la médiane
def remplacerValeursMinParMediane(dataframe, colonnes):
    for colonne in colonnes:
        # Calculer la moyenne et l'écart-type de la colonne
        moyenne = dataframe[colonne].mean()
        ecart_type = dataframe[colonne].std()
        
        # Définir une valeur seuil (Les valeurs qui se trouvent à plus de trois écarts-types de la moyenne sont considérées comme des valeurs aberrantes)
        seuil = moyenne + 3 * ecart_type
        
        # Calculer la médiane de la colonne
        mediane = dataframe[colonne].median()
        
        # Remplacer les valeurs supérieures au seuil par la médiane
        dataframe.loc[dataframe[colonne] < seuil, colonne] = mediane
    return dataframe

In [429]:
# Fonctions pour visualiser les données

## Fonction pour visualiser les valeurs manquantes
def visuelDonneesNa(df):
    msno.bar(df,  color= 'dodgerblue')

## Fontion qui renvoie un histogramme et un boxplot pour chaque variable quantitative
def histoBoxPlotQuanti(data, colonnes):
    for col in colonnes:
        plt.figure(figsize=(10, 4))
        
        # Histogramme avec estimation de la densité de noyau (Kernel Density Estimate)
        plt.subplot(1, 2, 1)
        sns.histplot(data[col], kde=True)
        plt.title(f'Histogramme de {col}')
        plt.xlabel(col)
        plt.ylabel('Fréquence')
        
        # Boxplot
        plt.subplot(1, 2, 2)
        sns.boxplot(y=data[col])
        plt.title(f'Boxplot de {col}')
        plt.ylabel(col)
        
        plt.tight_layout()
        plt.show()

In [430]:
#Importation du fichier
fichier = importFichier("2016_Building_Energy_Benchmarking.csv")
fichier.head()

In [431]:
descriptionData(fichier)

In [432]:
#Visualisation des valeurs manquantes
visuelDonneesNa(fichier)

In [433]:
# Sélection des colonnes avec un taux de remplissage < 50%
col_na = colManquantes(fichier, 50)
col_na

In [434]:
# Suppression des colonnes avec un taux de remplissage < 50%
fichier_col_supp_50 = supprimeCol(fichier, col_na)
fichier_col_supp_50.head()

In [435]:
# visuel du taux de remplissage
visuelDonneesNa(fichier_col_supp_50)

In [436]:
# Premier choix et description des varialbes
# Liste des noms des colonnes sélectionnées
variables_list = ['OSEBuildingID', 'BuildingType', 'PrimaryPropertyType', 'YearBuilt', 'NumberofBuildings', 'NumberofFloors',
                  'PropertyGFATotal', 'PropertyGFAParking', 'PropertyGFABuilding(s)', 'SiteEUI(kBtu/sf)', 'SiteEUIWN(kBtu/sf)', 
                  'SourceEUI(kBtu/sf)', 'SourceEUIWN(kBtu/sf)', 'SiteEnergyUse(kBtu)', 'SiteEnergyUseWN(kBtu)', 'SteamUse(kBtu)', 
                  'Electricity(kWh)', 'Electricity(kBtu)', 'NaturalGas(therms)', 'NaturalGas(kBtu)', 'GHGEmissionsIntensity', 
                  'ENERGYSTARScore']

# Liste des descriptions correspondantes
descriptions = ['ID du bâtiment OSE', 'Type de bâtiment (résidentiel, commercial, etc.)', 'Type de propriété principale', 'Année de construction', 'Nombre de bâtiments', 
                'Nombre d\'étages', 'Surface totale du bâtiment et du parking', 'Surface de stationnement de la propriété', 'Surface du bâtiment de la propriété', 
                'utilisation de l\'énergie du site d\'une propriété divisée par sa superficie brute au sol', 'EUI du site normalisé (énergétique du site que la propriété aurait consommée pendant les conditions météorologiques moyennes de 30 ans)',
                'énergie de source d\'une propriété divisée par sa surface brute au sol', 'EUI de la source normalisé (kBtu/sf)', 
                'Utilisation d\'énergie du site (kBtu)', 'Utilisation d\'énergie du site normalisé (kBtu)', 'La quantité annuelle de vapeur de district consommée par la propriété sur place, mesurée en milliers d\'unités thermiques britanniques (kBtu)', 
                'La quantité annuelle d\'électricité consommée par la propriété', 'La quantité annuelle d\'électricité consommée par la propriété', 'La quantité annuelle de gaz naturel consommée par la propriété (therms)', 'La quantité annuelle de gaz naturel consommée par la propriété (kBtu)', 
                'Intensité des émissions de GES (kgCO2e/ft2)', 'Score ENERGY STAR']

# Création de la DataFrame
df_description = pd.DataFrame({
    'Nom de la colonne': variables_list,
    'Description': descriptions
})

# Ajuster les paramètres d'affichage de pandas pour afficher toutes les colonnes
pd.set_option('display.max_columns', None) 
pd.set_option('display.max_colwidth', None)

# Affichage de la DataFrame
df_description

In [437]:
# Sélection des varialbes
data_select = colSelection(fichier_col_supp_50, variables_list)
data_select.head()

In [438]:
# Nombre bâtiment par type
data_select["BuildingType"].value_counts()

In [439]:
# Selection des bâtiments contenant le type "NonResidential"
data_select_filtre = data_select[data_select["BuildingType"].str.contains("(?i)NonResidential", na=False)]
data_select_filtre.head()

In [440]:
# Nombre bâtiment par type (vérification)
data_select_filtre["BuildingType"].value_counts()

In [441]:
# Conversion de l'id des batiments et de l'année de conctruction en facteur
data_select_filtre[["OSEBuildingID", "YearBuilt"]] = data_select_filtre[["OSEBuildingID", "YearBuilt"]].astype('category')

In [442]:
data_select_filtre.info()

In [443]:
# Catégorisation des variables
quali,quanti = categoriseVariables(data_select_filtre)

In [444]:
# variables qualitatives
quali

In [445]:
# variables quantitatives
quanti

In [446]:
#histoBoxPlotQuanti(fichier_col_supp_50, quanti)

In [447]:
# verification des doublons
data_select_filtre["OSEBuildingID"].duplicated().sum()

In [448]:
data_select_filtre.describe()

In [449]:
# Filtre des valeurs inférieures à 0
## selectionner les colonnes quantitatives
data_quanti = data_select_filtre[quanti]
## Filtrer les lignes où les valeurs des colonnes quantitatives sont inférieures à 0
negative_quanti_val = data_quanti[(data_quanti < 0).any(axis = 1)]
negative_quanti_val

In [450]:
# Suppression des valeurs inférieures à 0
data_select_filtre = data_select_filtre.drop(negative_quanti_val.index).reset_index(drop=True)
data_select_filtre.head()

In [451]:
# Choix des variables finales
quali = ['OSEBuildingID', 'YearBuilt']
quanti = ['NumberofBuildings', 'NumberofFloors', 'PropertyGFATotal', 'PropertyGFAParking', 'PropertyGFABuilding(s)', 'SiteEUI(kBtu/sf)', 'SiteEUIWN(kBtu/sf)',
         'SourceEUI(kBtu/sf)', 'SourceEUIWN(kBtu/sf)', 'SiteEnergyUse(kBtu)', 'SiteEnergyUseWN(kBtu)', 'SteamUse(kBtu)', 'Electricity(kBtu)', 'NaturalGas(kBtu)', 'GHGEmissionsIntensity', 'ENERGYSTARScore']

# Dataframe finale
data_final = data_select_filtre[quali + quanti]
data_final.head()

In [452]:
# Description de la dataframe finale
descriptionData(data_final)

In [453]:
# Description des variables finales
data_final_describe = df_description[df_description['Nom de la colonne'].isin(data_final.columns)].reset_index(drop=True)
data_final_describe

In [454]:
# Vérification des valeurs manquantes
data_final.isnull().sum()

In [455]:
# Remplacement des valeurs manquantes par la methode statistique appropriée
## selection des colonnes quantitatives avec des valeurs manquantes a remplacer
quanti_select = ['NumberofBuildings', 'NumberofFloors', 'PropertyGFATotal', 'PropertyGFAParking', 'PropertyGFABuilding(s)', 'SiteEUI(kBtu/sf)', 'SiteEUIWN(kBtu/sf)',
         'SourceEUI(kBtu/sf)', 'SourceEUIWN(kBtu/sf)', 'SiteEnergyUse(kBtu)', 'SiteEnergyUseWN(kBtu)', 'SteamUse(kBtu)', 'Electricity(kBtu)', 'NaturalGas(kBtu)', 'GHGEmissionsIntensity']
data_final = remplacerValParStats(data_final, quanti_select)

In [456]:
# Revérification des valeurs manquantes
data_final.isnull().sum()

In [457]:
# fonction qui retourne un boxplot des valeurs aberrantes des variables quantitatives  
def boxplotQuanti(data, colonnes):
    plt.boxplot([data[col].dropna() for col in colonnes ], labels= colonnes, vert= False)
    plt.title("Boxplot des variables quantitatives")
    plt.xlabel("Valeurs")
    plt.ylabel("Colonnes")
    plt.show()

# visualisation des valeurs aberrantes
boxplotQuanti(data_final, quanti)

In [458]:
# Description des variables quantitatives
data_final.describe()

In [459]:
# Remplacement des valeurs trop grandes (supérieures à 3 fois l'écart-type) par la médiane
data_final = remplacerValeursMaxParMediane(data_final, quanti_select)

In [460]:
data_final.describe()

In [461]:
# Nombre de valeurs À 0 présentes par colonne
data_final[data_final == 0].count() 

In [462]:
# Remplacement des valeurs trop petites (inférieur à 3 fois l'écart-type) par la médiane
data_final = remplacerValeursMinParMediane(data_final, quanti_select)

In [463]:
# Revérification du nombre de valeurs À 0 présentes par colonne
data_final[data_final == 0].count() 

In [464]:
descriptionData(data_final)

In [465]:
data_final["PropertyGFAParking"].value_counts()

In [466]:
data_final["SteamUse(kBtu)"].value_counts()

In [467]:
#Fonction pour remplacer les 0 par la médiane
def remplacerZeroParMediane(dataframe, colonnes):
    for colonne in colonnes:
        mediane = dataframe[colonne].median()
        dataframe.loc[dataframe[colonne] == 0, colonne] = mediane
    return dataframe

# Remplacement des valeurs à 0 par la médiane
data_final = remplacerZeroParMediane(data_final, quanti_select)

In [468]:
# dernière vérification Nombre de valeurs À 0 présentes par colonne
data_final[data_final == 0].count() 

Les valeurs à 0 des colonnes PropertyGFAParking (Surface de stationnement de la propriété) et SteamUse (La quantité annuelle de vapeur de district consommée par la propriété) n'ont pas changés, celà semble bizare mais pas impossible donc on va les laisser telquel .

In [469]:
# Détection des valeurs abérrantes
## Methode de l'intervalle interquartile des variables quantitatives
def detectOutlierIqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    born_inf = Q1 - 1.5 * IQR
    born_sup = Q3 + 1.5 * IQR
    outliers = data[(data[column] < born_inf) | (data[column] > born_sup)]
    return outliers

outliers = detectOutlierIqr(data_final, "SiteEnergyUseWN(kBtu)")
outliers

In [470]:
# Description des valeurs abérrantes
descriptionData(outliers)

In [471]:
# Suppression des valeurs aberrantes avec IQR
def remove_outliers(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return data[(data[column] >= lower_bound) & (data[column] <= upper_bound)]

# Exemple d'application
data_clean = remove_outliers(data_final, 'SiteEnergyUseWN(kBtu)')

In [472]:
data_clean["ENERGYSTARScore"].unique()

In [473]:
data_clean.head()

In [474]:
# Transformation des colonnes en proportions par rapport à la consommation totale d'énergie
def transformer_en_proportions(data):
    # Calcul de la consommation totale d'énergie
    data['TotalEnergy(kBtu)'] = data['Electricity(kBtu)'] + data['SteamUse(kBtu)'] + data['NaturalGas(kBtu)']
    
    # Transformation des colonnes en proportions
    data['Electricity_Proportion'] = data['Electricity(kBtu)'] / data['TotalEnergy(kBtu)']
    data['SteamUse_Proportion'] = data['SteamUse(kBtu)'] / data['TotalEnergy(kBtu)']
    data['NaturalGas_Proportion'] = data['NaturalGas(kBtu)'] / data['TotalEnergy(kBtu)']
    
    return data

# Application de la transformation
data_clean = transformer_en_proportions(data_clean)

# Affichage des premières lignes pour vérifier
data_clean[['Electricity_Proportion', 'SteamUse_Proportion', 'NaturalGas_Proportion']].head()

In [475]:
# Suppression des colonnes de consommation d'énergie brutes
data_clean = data_clean.drop(columns=['Electricity(kBtu)', 'SteamUse(kBtu)', 'NaturalGas(kBtu)', 'TotalEnergy(kBtu)'])

In [485]:
# dataframe finale
data_clean.head()

### Analyses

##### UNIVARIEE

In [478]:
# Nouvelle atégorisation des variables
quali,quanti = categoriseVariables(data_clean)

In [479]:
# Visualisation des variables catégorielles
histoBoxPlotQuanti(data_clean, quali)

In [480]:
# Visualisation des variables continues
histoBoxPlotQuanti(data_clean, quanti)

##### BIVARIEE

In [482]:
sns.pairplot(data_clean[quanti])

In [484]:
def heatmap(data, colonnes, check_normality=False):
    if check_normality:
        for col in colonnes:
            stat, p = st.shapiro(data[col])
            if p <= 0.05:
                print(f"La colonne {col} ne suit pas une distribution normale (p={p:.3f}).")
    
    # Calculer la matrice de corrélation
    corr_matrix = data[colonnes].corr()
    
    # Tracer la carte thermique
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr_matrix, annot=True, cmap='YlGnBu', vmin=-1, vmax=1)
    plt.title("Matrice de corrélation")
    plt.show()

heatmap(data_clean, quanti)